<a href="https://colab.research.google.com/github/michaelseth5/Accenture-1K-contract-review-challenge/blob/main/cuad_issue_19_end_to_end_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CUAD contract-review pipeline | Issue #19

This notebook connects **multi-label clause predictions → source evidence windows → explainable provisional risk rules → contract-level triage**.

**Important:** The real-model step requires a **fine-tuned CUAD model checkpoint**, not a base DistilRoBERTa model or the two-example demonstration checkpoint. It also requires an ordered list of labels matching the model's output neurons.

The notebook first tests the wiring with synthetic data; then optionally runs a real checkpoint on the official CUAD `test.json` (102 contracts). The mock test is *not* evidence of model quality. Do not upload private contracts into a public repository.

## 1. Set up Colab

Choose **Runtime → Change runtime type → T4 GPU**, if available. CPU works but can be slow for 102 full-length contracts.

In [ ]:
!pip -q install "transformers>=4.45,<5" torch

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


## 2. Write the reusable pipeline module

In [ ]:
from pathlib import Path
Path('contract_pipeline.py').write_text('"""CUAD clause -> evidence -> explainable risk -> contract-level triage.\n\nNo model weights or risk ground truth are bundled. Run with a fine-tuned\nmulti-label HF checkpoint and its ordered label mapping. Risk rules are\nillustrative and need advisor calibration before operational use.\n"""\nfrom __future__ import annotations\nimport argparse\nimport json\nimport math\nimport re\nfrom pathlib import Path\nfrom typing import Callable\n\n# Provisional category-level triage weights, NOT CUAD risk annotations.\nPRIORITY = {\n    "Uncapped Liability": 5, "Cap On Liability": 4, "Indemnification": 4,\n    "Termination For Convenience": 3, "Auto-Renewal": 3,\n    "IP Ownership Assignment": 4, "Change Of Control": 3,\n    "Non-Compete": 3, "Audit Rights": 2, "Most Favored Nation": 3,\n}\nMANDATORY_REVIEW = ("Cap On Liability", "Indemnification", "IP Ownership Assignment")\nRISK_FLAGS = {\n    "uncapped_exposure": re.compile(r"\\b(unlimited liability|uncapped liability|without limit(?:ation)? of liability)\\b", re.I),\n    "unilateral_termination": re.compile(r"\\b(sole discretion|at its sole option)\\b", re.I),\n    "automatic_renewal": re.compile(r"\\b(automatically renew|automatic renewal|auto.renew)\\b", re.I),\n}\n\n\ndef load_cuad(path: str | Path):\n    """Yield contract IDs and text from CUAD SQuAD-style JSON (not annotations)."""\n    with open(path, encoding="utf-8") as f:\n        data = json.load(f)\n    if not isinstance(data.get("data"), list):\n        raise ValueError("Expected CUAD JSON with a data list")\n    for index, item in enumerate(data["data"]):\n        title = item.get("title") or f"contract-{index}"\n        paragraphs = item.get("paragraphs", [])\n        text = "\\n\\n".join(p.get("context", "") for p in paragraphs if p.get("context"))\n        if not text.strip():\n            raise ValueError(f"Empty contract: {title}")\n        yield {"contract_id": str(title), "text": text}\n\n\ndef chunk_text(text: str, tokenizer, max_tokens: int = 224, overlap: int = 48):\n    """Token-based windows with exact source character offsets."""\n    if not (0 <= overlap < max_tokens):\n        raise ValueError("Require 0 <= overlap < max_tokens")\n    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True,\n                        truncation=False)\n    offsets = encoded["offset_mapping"]\n    step = max_tokens - overlap\n    for start in range(0, len(offsets), step):\n        stop = min(start + max_tokens, len(offsets))\n        first, last = offsets[start][0], offsets[stop - 1][1]\n        if first >= last:\n            continue\n        yield {"start": int(first), "end": int(last), "text": text[first:last]}\n        if stop == len(offsets):\n            break\n\n\ndef hf_predictor(checkpoint: str, labels: list[str], device: str | None = None):\n    """Return (tokenizer, predictor). Checkpoint MUST be task fine-tuned."""\n    import torch\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)\n    if not tokenizer.is_fast:\n        raise ValueError("Evidence offsets require a fast tokenizer")\n    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)\n    if model.config.num_labels != len(labels):\n        raise ValueError("Checkpoint output dimension and ordered labels differ")\n    device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    model.to(device).eval()\n\n    def predict(batch: list[str]) -> list[list[float]]:\n        inputs = tokenizer(batch, return_tensors="pt", padding=True,\n                           truncation=True, max_length=256).to(device)\n        with torch.inference_mode():\n            scores = torch.sigmoid(model(**inputs).logits).cpu().tolist()\n        return scores\n    return tokenizer, predict\n\n\ndef score_clause(category: str, evidence: str) -> dict:\n    """Transparent, provisional heuristic; category presence is not automatically risky."""\n    matched = [name for name, pattern in RISK_FLAGS.items() if pattern.search(evidence)]\n    points = PRIORITY.get(category, 1)\n    if category == "Uncapped Liability":\n        points = 5\n    if "uncapped_exposure" in matched:\n        points = 5\n    elif matched:\n        points = min(5, points + 1)\n    level = "High" if points >= 5 else "Medium" if points >= 3 else "Low"\n    return {"risk_score": points, "risk_level": level,\n            "rationale": f"Category weight {PRIORITY.get(category, 1)}; " +\n                         ("matched indicators: " + ", ".join(matched) if matched else "no rule indicators")}\n\n\ndef analyze_contract(contract: dict, labels: list[str], tokenizer,\n                     predict: Callable[[list[str]], list[list[float]]],\n                     threshold: float = 0.5, batch_size: int = 8) -> dict:\n    if not 0 <= threshold <= 1:\n        raise ValueError("Threshold must be between 0 and 1")\n    if not labels or len(labels) != len(set(labels)):\n        raise ValueError("Ordered labels must be nonempty and unique")\n    text = contract["text"]\n    chunks = list(chunk_text(text, tokenizer))\n    evidence = {}\n    for pos in range(0, len(chunks), batch_size):\n        subset = chunks[pos:pos + batch_size]\n        outputs = predict([c["text"] for c in subset])\n        if len(outputs) != len(subset):\n            raise ValueError("Predictor returned wrong batch size")\n        for chunk, probs in zip(subset, outputs):\n            if len(probs) != len(labels):\n                raise ValueError("Predictor returned wrong category count")\n            for category, prob in zip(labels, probs):\n                prob = float(prob)\n                if not math.isfinite(prob) or not 0 <= prob <= 1:\n                    raise ValueError("Invalid prediction probability")\n                if prob < threshold:\n                    continue\n                # Overlapping windows may predict the same category; retain the\n                # most confident source-backed candidate per category.\n                if category not in evidence or prob > evidence[category]["confidence"]:\n                    evidence[category] = {"category": category,\n                        "confidence": round(prob, 5), "evidence": chunk["text"],\n                        "start": chunk["start"], "end": chunk["end"]}\n    clauses = []\n    for category, candidate in evidence.items():\n        clauses.append({**candidate, **score_clause(category, candidate["evidence"])})\n    clauses.sort(key=lambda x: (-x["risk_score"], -x["confidence"], x["category"]))\n    # Missing-category signals indicate REVIEW of uncertain absence, not\n    # a verified absence; low recall makes unqualified claims unsafe.\n    unchecked = [category for category in MANDATORY_REVIEW if category in labels and category not in evidence]\n    absent_penalty = 4 if unchecked else 0\n    highest = max([c["risk_score"] for c in clauses] + [absent_penalty])\n    return {\n        "contract_id": contract["contract_id"], "risk_score": highest,\n        "risk_level": "High" if highest >= 5 else "Medium" if highest >= 3 else "Low",\n        "review_priority": "Urgent" if highest >= 5 else "Review" if highest >= 3 else "Routine",\n        "clause_count": len(clauses), "clauses": clauses,\n        "unverified_missing_priority_categories": unchecked,\n        "limitations": "Evidence is the source text window, not a precise clause span. "\n                       "Negative predictions cannot establish clause absence. "\n                       "Risk rules are provisional, not legally validated."\n    }\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--cuad", required=True, help="CUAD SQuAD-style JSON")\n    parser.add_argument("--checkpoint", required=True, help="Fine-tuned HF checkpoint directory")\n    parser.add_argument("--labels", required=True, help="Ordered JSON array of model categories")\n    parser.add_argument("--output", required=True, help="Output JSONL path")\n    parser.add_argument("--threshold", type=float, default=0.5)\n    parser.add_argument("--limit", type=int, default=0, help="0 = all contracts")\n    args = parser.parse_args()\n    labels = json.loads(Path(args.labels).read_text(encoding="utf-8"))\n    tokenizer, predict = hf_predictor(args.checkpoint, labels)\n    count = 0\n    with open(args.output, "w", encoding="utf-8") as output:\n        for contract in load_cuad(args.cuad):\n            result = analyze_contract(contract, labels, tokenizer, predict, args.threshold)\n            output.write(json.dumps(result, ensure_ascii=False) + "\\n")\n            count += 1\n            if args.limit and count >= args.limit:\n                break\n    if not count:\n        raise RuntimeError("No CUAD contracts found")\n    print(f"Processed {count} contracts -> {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Created contract_pipeline.py')

Created contract_pipeline.py


## 3. Write and run the unit tests

In [ ]:
from pathlib import Path
Path('test_contract_pipeline.py').write_text('import json\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\nimport unittest\nfrom contract_pipeline import load_cuad, chunk_text, analyze_contract\n\n\nclass FakeTokenizer:\n    def __call__(self, text, **kwargs):\n        import re\n        return {"offset_mapping": [(m.start(), m.end()) for m in re.finditer(r"\\S+", text)]}\n\n\nclass PipelineTests(unittest.TestCase):\n    def test_cuad_parsing_and_evidence_and_risk(self):\n        snippet = "The parties agree to unlimited liability for losses."\n        with TemporaryDirectory() as tmp:\n            path = Path(tmp) / "test.json"\n            path.write_text(json.dumps({"version": "1.0", "data": [\n                {"title": "Contract A", "paragraphs": [{"context": snippet, "qas": []}]},\n                {"title": "Contract B", "paragraphs": [{"context": "No agreement terms here.", "qas": []}]}\n            ]}), encoding="utf-8")\n            contracts = list(load_cuad(path))\n        self.assertEqual(len(contracts), 2)\n        labels = ["Uncapped Liability", "Indemnification"]\n        def predict(batch):\n            return [[0.95, 0.2] if "unlimited liability" in x else [0.05, 0.1] for x in batch]\n        r = analyze_contract(contracts[0], labels, FakeTokenizer(), predict)\n        self.assertEqual(r["risk_level"], "High")\n        self.assertEqual(r["review_priority"], "Urgent")\n        self.assertEqual(r["clause_count"], 1)\n        clause = r["clauses"][0]\n        self.assertEqual(snippet[clause["start"]:clause["end"]], clause["evidence"])\n        self.assertIn("Indemnification", r["unverified_missing_priority_categories"])\n        b = analyze_contract(contracts[1], labels, FakeTokenizer(), predict)\n        self.assertEqual(b["risk_level"], "Medium")\n        self.assertEqual(b["clause_count"], 0)\n\n    def test_overlap_preserves_offsets(self):\n        text = "alpha beta gamma delta epsilon zeta eta"\n        windows = list(chunk_text(text, FakeTokenizer(), max_tokens=3, overlap=1))\n        self.assertEqual(len(windows), 3)\n        self.assertTrue(all(text[w["start"]:w["end"]] == w["text"] for w in windows))\n\n    def test_reject_bad_model_shape(self):\n        with self.assertRaisesRegex(ValueError, "category count"):\n            analyze_contract({"contract_id":"x", "text":"hello"}, ["A", "B"],\n                             FakeTokenizer(), lambda _: [[0.5]])\n\n\nif __name__ == "__main__":\n    unittest.main()\n', encoding='utf-8')
!python -m unittest -v test_contract_pipeline.py

test_cuad_parsing_and_evidence_and_risk (test_contract_pipeline.PipelineTests.test_cuad_parsing_and_evidence_and_risk) ... ok
test_overlap_preserves_offsets (test_contract_pipeline.PipelineTests.test_overlap_preserves_offsets) ... ok
test_reject_bad_model_shape (test_contract_pipeline.PipelineTests.test_reject_bad_model_shape) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.002s

OK


## 4. Download the actual CUAD held-out contracts

This uses the project's `data/test.json` source. If GitHub's raw download is unavailable, upload `test.json` manually from the repository's `data/` folder. The dataset contains annotated legal contracts and should be attributed to The Atticus Project (CC BY 4.0).

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
url = "https://raw.githubusercontent.com/Break-Through-Tech/Accenture-1K-contract-review-challenge/main/data/test.json"
target = Path("test.json")
try:
    urlretrieve(url, target)
except Exception as exc:
    print("Automatic download failed:", exc)
    from google.colab import files
    files.upload()
assert target.exists(), "Upload the CUAD data/test.json file to Colab"
from contract_pipeline import load_cuad
contracts = list(load_cuad(target))
print("Loaded contracts:", len(contracts))
assert len(contracts) == 102, f"Expected 102 official test contracts, found {len(contracts)}"

Loaded contracts: 102


## 5. Supply the **fine-tuned** checkpoint and label list

You can mount Google Drive or upload a ZIP containing your checkpoint. Expected model directory: `config.json`, weights (`model.safetensors` or `pytorch_model.bin`), and fast-tokenizer files. The `ordered_labels.json` file must contain a JSON array of the exact 41 category names **in the same index order used when the model was trained**.

Example: `["Document Name", "Parties", ...]` is **illustrative only** — do not use a guessed label order. If your model checkpoint uses a different subset of classes, provide that exact ordered mapping.

**If you don't have the trained model + mapping, stop here:** unit tests show the interface works, but actual CUAD inference cannot be validated.

In [ ]:

from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from pathlib import Path
import io, json, zipfile

# Authenticate to access the shared BTT Drive file
auth.authenticate_user()
service = build("drive", "v3")

file_id = "1dx6eKI09MaWnDJN5IlJ4FGW6dDHUXpXF"
zip_path = Path("/content/CUAD_Classifier_v1.zip")
extract_dir = Path("/content/CUAD_Classifier_v1")

request = service.files().get_media(fileId=file_id)

with zip_path.open("wb") as output:
    downloader = MediaIoBaseDownload(output, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(f"Downloaded {status.progress():.0%}")

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(extract_dir)

# Load the exact category order from training
info = json.loads(
    (extract_dir / "model_info.json").read_text()
)

CHECKPOINT_DIR = str(extract_dir / "model")
labels = info["categories"]

assert (Path(CHECKPOINT_DIR) / "config.json").exists()
assert len(labels) == info["num_categories"]

print("Model:", CHECKPOINT_DIR)
print("Number of categories:", len(labels))
print("Model loaded and ready for inference setup!")


Downloaded 34%
Downloaded 68%
Downloaded 100%
Model: /content/CUAD_Classifier_v1/model
Number of categories: 41
Model loaded and ready for inference setup!


## 6. Smoke-test 5 real CUAD contracts

This invokes the checkpoint, preserves offsets into source text, and checks that the model outputs valid probabilities. Review the first clause's evidence *before* running all 102 contracts.

In [ ]:
from contract_pipeline import hf_predictor, analyze_contract
tokenizer, predict = hf_predictor(CHECKPOINT_DIR, labels)
smoke_results = [
    analyze_contract(contract, labels, tokenizer, predict, threshold=0.5)
    for contract in contracts[:5]
]
assert len(smoke_results) == 5
for contract, result in zip(contracts[:5], smoke_results):
    for clause in result["clauses"]:
        assert contract["text"][clause["start"]:clause["end"]] == clause["evidence"]
print("Smoke test completed on", len(smoke_results), "CUAD contracts.")
print(json.dumps(smoke_results[0], indent=2, ensure_ascii=False)[:4000])

Token indices sequence length is longer than the specified maximum sequence length for this model (2548 > 512). Running this sequence through the model will result in indexing errors


Smoke test completed on 5 CUAD contracts.
{
  "contract_id": "LohaCompanyltd_20191209_F-1_EX-10.16_11917878_EX-10.16_Supply Agreement",
  "risk_score": 4,
  "risk_level": "Medium",
  "review_priority": "Review",
  "clause_count": 6,
  "clauses": [
    {
      "category": "Parties",
      "confidence": 0.925,
      "evidence": "Exhibit 10.16 SUPPLY CONTRACT Contract No: Date: The buyer/End-User: Shenzhen LOHAS Supply Chain Management Co., Ltd. ADD: Tel No. : Fax No. : The seller: ADD: The Contract is concluded and signed by the Buyer and Seller on , in Hong Kong. 1. General provisions 1.1 This is a framework agreement, the terms and conditions are applied to all purchase orders which signed by this agreement (hereinafter referred to as the \"order\"). 1.2 If the provisions of the agreement are inconsistent with the order, the order shall prevail. Not stated in order content will be subject to the provisions of agreement. Any modification, supplementary, give up should been written recor

## 7. Run the full held-out CUAD set and export contract reports

This checks the pipeline's ability to process **all 102 test contracts**. It does **not** automatically establish clause precision/recall/F1 or risk validity; those require separate evaluation against CUAD labels and advisor-rated risk examples.

In [ ]:
from pathlib import Path
results = []
out = Path("cuad_contract_reports.jsonl")
with out.open("w", encoding="utf-8") as fh:
    for index, contract in enumerate(contracts, start=1):
        result = analyze_contract(contract, labels, tokenizer, predict, threshold=0.5)
        for clause in result["clauses"]:
            assert contract["text"][clause["start"]:clause["end"]] == clause["evidence"]
        fh.write(json.dumps(result, ensure_ascii=False) + "\n")
        results.append(result)
        if index % 10 == 0:
            print(f"Processed {index}/{len(contracts)}")
assert len(results) == 102
print("Success:", len(results), "contract reports saved to", out)

Processed 10/102
Processed 20/102
Processed 30/102
Processed 40/102
Processed 50/102
Processed 60/102
Processed 70/102
Processed 80/102
Processed 90/102
Processed 100/102
Success: 102 contract reports saved to cuad_contract_reports.jsonl


## 8. View a summary and download GitHub-ready artifacts

The risk score is a **provisional triage heuristic**, not a legal judgment. The evidence is the *winning source window* rather than a precise annotated clause span. Missing high-priority categories are marked **unverified**, not proven absent.

In [ ]:
import pandas as pd
summary = pd.DataFrame([
    {"contract_id": r["contract_id"], "risk_score": r["risk_score"],
     "risk_level": r["risk_level"], "review_priority": r["review_priority"],
     "clause_count": r["clause_count"]}
    for r in results
])
display(summary.head(20))
print(summary["risk_level"].value_counts(dropna=False))
summary.to_csv("cuad_contract_summary.csv", index=False)
from google.colab import files
files.download("cuad_contract_reports.jsonl")
files.download("cuad_contract_summary.csv")
files.download("contract_pipeline.py")
files.download("test_contract_pipeline.py")

,contract_id,risk_score,risk_level,review_priority,clause_count
0,LohaCompanyltd_20191209_F-1_EX-10.16_11917878_...,4,Medium,Review,6
1,CENTRACKINTERNATIONALINC_10_29_1999-EX-10.3-WE...,4,Medium,Review,9
2,DovaPharmaceuticalsInc_20181108_10-Q_EX-10.2_1...,4,Medium,Review,17
3,"PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC L...",4,Medium,Review,18
4,ThriventVariableInsuranceAccountB_20190701_N-6...,4,Medium,Review,4
5,"HERTZGLOBALHOLDINGS,INC_07_07_2016-EX-10.4-INT...",4,Medium,Review,10
6,TUNIUCORP_03_06_2014-EX-10-COOPERATION AGREEMENT,4,Medium,Review,8
7,CYBERIANOUTPOSTINC_07_09_1998-EX-10.13-PROMOTI...,4,Medium,Review,15
8,MSCIINC_02_28_2008-EX-10.10-,4,Medium,Review,9
9,CHERRYHILLMORTGAGEINVESTMENTCORP_09_26_2013-EX...,4,Medium,Review,7


risk_level
Medium    102
Name: count, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Add results to GitHub manually

Commit `contract_pipeline.py`, `test_contract_pipeline.py`, and this notebook under your team's preferred project paths. Review the report before committing any generated data; large/full contract-text evidence may be unsuitable for GitHub.

For Issue #19, document:
- Number of CUAD test contracts actually processed (target: 102)
- Checkpoint and label-map version, inference threshold, chunking policy, and test runtime
- Any errors and representative source-backed evidence windows
- Explicit limitation: CUAD has no risk-level ground truth; request advisor validation for triage rules
- Separate clause-level metrics and risk-ranking metrics when those evaluations are available

Do **not** mark Issue #19 complete merely because the synthetic unit tests passed.